# Synthetic RAG Data Generation

This notebook walks through the steps to automatically generate question-answer pairs for training a search / RAG model from a data corpus.

Data corpus can be any collection of text documents, such as:

- markdown files (notion / developer docs / obsidian vault)
- pdfs
- emails
- text / slack messages
- mix of various sources

This notebook currently only supports parsing markdown files. Support for other modalities will be added with time.

The main process for generating synthetic data for search / RAG:

1. Sample a section of a document as a reference chunk (ideally it contains interesting information)
2. Tell an LLM to generate a reasonable search query + answer pair that could only be found in this sample.

With this process, we will get a list of <question, answer, reference chunk(s)> which will serve as the train / eval dataset.
Reference chunk can be used in the reward function to determine if the search agent has retrieved the proper context.

This notebook covers the basics to get a decent v1 dataset - feel free to extend this notebook to generate / filter for higher quality questions :)


## 0. Setup

### Google Colab Setup

This section sets up the environment for running in Google Colab:

1. Clone the synthetic-data-prep library
2. Install dependencies
3. Configure API credentials


In [ ]:
# Clone the repository
# !git clone https://github.com/cgft-io/synthetic-data-prep-lib.git
# %cd synthetic-data-prep-lib

# # Install the library with dev dependencies
# %pip install -e .[dev]

In [1]:
# This is only used internally for development

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Add the src directory to the path for local imports
repo_root = Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

## 1. Document Chunking

RAG systems retrieve chunks, not entire documents. So our first step is splitting documents into pieces that can be indexed and retrieved.

We went with chunk size of min 1024 chars with a max size of 2048 as a reasonable starting point, though you should experiment based on your use case.


In [2]:
from synthetic_data_prep.chunkers.inspector import ChunkInspector
from synthetic_data_prep.chunkers.markdown import MarkdownChunker

# ===== MODIFY HERE: Replace the placeholder / tweak the configs =====
DOCS_PATH = "./samples/posthog"
# DOCS_PATH = "./samples/cgft-notion"
# DOCS_PATH = "./samples/based.cooking"
# DOCS_PATH = "./samples/learning-notes"

# Our markdown chunker break down a document into chunks by the header
# If a chunk is too small (< MIN_CHARS), we will try to fuse the chunk
# with its neighbor while not exceeding MAX_CHARS
# If a chunk is too big (> MAX_CHARS), we will break it down
# into smaller chunks while using OVERLAP_CHARS to retain context of the prev chunk.

MIN_CHARS = 1024
MAX_CHARS = 2048
OVERLAP_CHARS = 128
# ===== END OF MODIFY SECTION =====

# Configure chunker
chunker = MarkdownChunker(min_char=MIN_CHARS, max_char=MAX_CHARS, chunk_overlap=OVERLAP_CHARS)

# Chunk folder - returns ChunkCollection
collection = chunker.chunk_folder(DOCS_PATH, file_extensions=[".md", ".mdx"])

# Use inspector to analyze chunks
inspector = ChunkInspector(collection)
inspector.summary(max_depth=3, max_files_per_folder=4)

ChunkCollection Summary
Total chunks: 3165
Total files: 1176

Chunk Size Statistics:
  Min: 29 chars (docs/contribute/badge.md, chunk 0)
  Max: 2048 chars (docs/data-warehouse/query.mdx, chunk 0)
  Mean: 1148 chars

File Structure:
└── docs/
    ├── data-warehouse/
    │   ├── _snippets/
    │   ├── sources/
    │   ├── views/
    │   ├── sql/
    │   ├── under-the-hood.md (1 chunks)
    │   ├── cutting-costs.mdx (3 chunks)
    │   ├── join.mdx (2 chunks)
    │   ├── tutorials.mdx (1 chunks)
    │       ... 5 more files
    ├── advanced/
    │   ├── proxy/
    │   ├── _snippets/
    │   ├── content-security-policy.md (6 chunks)
    │   ├── cdp.md (4 chunks)
    │   ├── browser-extension.md (10 chunks)
    │   └── proxy.mdx (2 chunks)
    ├── cdp/
    │   ├── transformations/
    │   ├── destinations/
    │   ├── batch-exports/
    │   ├── sources/
    │   ├── _snippets/
    │   ├── troubleshooting.md (4 chunks)
    │   ├── fivetran-airbyte.md (3 chunks)
    │   ├── index.md (2 chunks)


In [ ]:
# UNCOMMENT the code below to inspect the quality of the chunking

# # Inspect a random chunk
# print(inspector.sample_chunk(show_context_before=True, show_context_after=True))

# # Inspect a random file
# print(inspector.sample_file(max_chunks=10))

# # Inspect a specific file
# print(inspector.read_file("product-toolkit.mdx", max_chunks=10))


# UNCOMMENT the code below to save chunks to a file
# save_chunks(collection, "outputs/chunks.yaml")

# UNCOMMENT the code below to load chunks from a file
# collection = load_chunks("outputs/chunks.yaml")
# inspector = ChunkInspector(collection)
# inspector.summary(max_depth=3, max_files_per_folder=4)

Feel free to tweak the configuration above to achieve better chunking


## 2. Upload Corpus

Upload your chunks to our corpus API for BM25 search and evaluation. This enables:

- Fast BM25 search over your chunked documents
- Integration with the training pipeline
- Evaluation of retrieval quality

**Note:** Chunks cannot be deleted individually. If you need to redo the upload, delete the corpus and create a new one.


In [ ]:
from synthetic_data_prep.corpus.client import CorpusClient

# ===== MODIFY HERE: Configure your API and corpus settings =====
API_KEY = "your-api-key"
CORPUS_NAME = "my-docs"
CORPUS_BASE_URL = "https://app.cgft.io"
# ===== END OF MODIFY SECTION =====

corpus_client = CorpusClient(api_key=API_KEY, base_url=CORPUS_BASE_URL)

# Get or create corpus (will prompt if limit reached)
corpus = corpus_client.get_or_create_corpus(CORPUS_NAME, on_limit="prompt")
print(f"Using corpus: {corpus.name} (ID: {corpus.id})")

# # UNCOMMENT the code below to delete an existing corpus
# corpus_client.delete_corpus(corpus.id)

In [ ]:
# Upload all chunks to the corpus
print(f"Uploading {len(collection)} chunks to corpus...")

upload_result = corpus_client.upload_chunks(
    corpus_id=corpus.id,
    collection=collection,
    batch_size=100,
    show_progress=True,
)

print("\nUpload complete!")
print(f"  Inserted: {upload_result.inserted_count}")
print(f"  Total chunks: {len(upload_result.chunk_ids)}")

In [ ]:
# Test the BM25 search functionality

# ===== MODIFY HERE: Test your search queries =====
TEST_QUERIES = [
    "how to feature flag",
    "setup reverse proxy",
    "nextjs installation",
]
# ===== END OF MODIFY SECTION =====

print("Testing search queries:\n")
for query in TEST_QUERIES:
    results = corpus_client.search(corpus_id=corpus.id, query=query, limit=3)

    print(f"Query: '{query}'")
    print(f"Found {results.total} results")

    for i, chunk_a in enumerate(results.results[:3], 1):
        preview = chunk_a.content[:100].replace("\n", " ")
        print(f"  {i}. (score: {chunk_a.score:.3f}) {preview}...")
    print()

## 3. Q&A Generation

We want to generate diverse and realistic questions that test different retrieval and reasoning capabilities. We group the questions into 2 main types:

**Single-Hop Questions** — One chunk contains the complete answer

- `postgres jsonb index syntax` -> SQL reference doc
- `company pto policy parental leave` -> HR handbook section
- `sourdough starter feeding ratio` -> recipe instructions

**Multi-Hop Questions** — Information from multiple chunks required

- `error handling in tanstack query mutations` -> TanStack Query overview -> mutations guide -> error boundary section
- `who approved the Q2 marketing budget` -> budget doc -> approval workflow -> executive sign-off

We'll tackle each question type in order, validating as we go.


### 3.1 LLM Client Setup


In [ ]:
from openai import OpenAI

# ===== MODIFY HERE: Configure your LLM settings =====
# Note: We provide access to a few standard models + some budget for you but feel free to BYOM
LLM_API_KEY = API_KEY
MODEL_ENDPOINT = "http://localhost:3000/api/llm"
MODEL = "grok-4-fast-non-reasoning"  # this model is the fastest with decent quality for our labelling task
# ===== END OF MODIFY SECTION =====

client = OpenAI(base_url=MODEL_ENDPOINT, api_key=LLM_API_KEY)

### 3.2 Generate Corpus Context

To create a corpus-level context, we can get the LLM to summarize the corpus using some top level / indexing chunks + sample a few random chunks.

It is also helpful for us (owner of the corpus) to provide some short context and example queries.


In [ ]:
import random

from synthetic_data_prep.chunkers.models import ChunkCollection
from synthetic_data_prep.qa_generation.helpers import (
    filter_chunks_by_length,
    print_sections,
    render_template,
)
from synthetic_data_prep.qa_generation.response_parsers import parse_corpus_summary_response

# ===== MODIFY HERE: Give your short description of the corpus and some example questions =====
YOUR_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]

NUM_TOP_LEVEL_CHUNK_SAMPLES = 4
NUM_RANDOM_SAMPLES = 4
# ===== END OF MODIFY SECTION =====

# CORPUS Analysis System and User prompts
corpus_system_prompt = """You are a technical analyst specializing in document corpus analysis.
Your goal is to understand the overall themes, content type, and typical query that a user might search for.

Based on the context and thoughts, form an understanding of the corpus, and then provide a short summary and example search queries. Weigh the user's input if provided.

Guideline:
- Try to generalize and identify the overall theme of the corpus.
- Use your context of the theme, domain, etc. to guess the persona and purpose of searching this corpus
- i.e.
    - student finding their learning notes
    - developer looking up API documentation
    - journalist researching a topic
    - business analyst gathering market research
- Use that persona to guess the types of queries they might perform.

For the summary:
- First line - corpus themes (documentation, tutorials, reference, etc.)
- Second line - content domain (technical, business, scientific, etc.)
- Third line - user persona and purpose (likely developer looking up API documentation)
- Do NOT cite specific chunk content in the summary.

For the example queries:
- Provide 5-10 realistic example queries a user might search for in this corpus.
- Use the inferred user persona and purpose to guide the query style.
- Queries can have incomplete information, as often users do not remember full context.

Return JSON with:
- thoughts: Your analysis and reasoning here
- summary: our summary here (3 lines as described above)
- example_queries: List of example queries in the form of ["query1", "query2", ...]

"""

corpus_user_template = """Analyze the following document corpus:

<user_context>
{user_context}
</user_context>

<top_level_chunks>
{top_level_content}
</top_level_chunks>

<random_sampled_chunks>
{random_content}
</random_sampled_chunks>

Return your analysis as JSON with keys: thoughts, summary, example_queries
"""


def generate_summary_and_queries(
    collection: ChunkCollection,
    your_description: str,
    example_queries: list[str],
    num_top_level: int,
    num_random: int,
) -> tuple[str, str, str]:
    all_chunks = filter_chunks_by_length(collection.chunks, min_chars=400)

    sampled_top_level = random.sample(collection.get_top_level_chunks(), num_top_level)
    sampled_random = random.sample(all_chunks, num_random)

    variables = {
        "user_context": f"Description: {your_description}\nExample queries provided by user: {', '.join(example_queries)}",
        "top_level_content": "\n\n".join([chunk.chunk_str() for chunk in sampled_top_level]),
        "random_content": "\n\n".join([chunk.chunk_str() for chunk in sampled_random]),
    }

    corpus_user_prompt = render_template(corpus_user_template, variables)

    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": corpus_system_prompt},
            {"role": "user", "content": corpus_user_prompt},
        ],
    )

    return corpus_system_prompt, corpus_user_prompt, completion.choices[0].message.content or ""


print("Generating corpus summary and example queries. This should take ~10+ seconds...\n")

system_prompt, user_prompt, response = generate_summary_and_queries(
    collection,
    YOUR_DESCRIPTION,
    EXAMPLE_QUERIES,
    NUM_TOP_LEVEL_CHUNK_SAMPLES,
    NUM_RANDOM_SAMPLES,
)

print_sections(
    # # Uncomment the following to inspect the system / user prompt
    # ( "System Prompt", system_prompt ),
    # ("User Prompt", user_prompt),
    ("Response", response),
)

corpus_summary, example_queries = parse_corpus_summary_response(response)
corpus_queries = "\n".join([f"- {q}" for q in example_queries])
corpus_context = {"corpus_summary": corpus_summary, "corpus_queries": corpus_queries}

### 3.3 Single-Hop Q&A Generation

Single-hop questions are the simplest to work with. All you need is a single reference chunk that contains meaningful information, and you can ask the model to generate query <> answer pairs.

On top of the reference chunk, providing corpus-level summary + example questions also helps with creating realistic questions.


In [ ]:
from synthetic_data_prep.chunkers.models import Chunk
from synthetic_data_prep.qa_generation.response_parsers import parse_single_hop_response

# ===== MODIFY HERE: feel free to adjust the num preview chars for the prev / after context =====
CONTEXT_PREVIEW_CHARS = 200
# ===== END OF MODIFY SECTION =====

single_hop_system_template = """You are generating realistic search queries for a RAG system.

Corpus summary:
{corpus_summary}

Example queries:
{corpus_queries}

Guide:
- When generating queries, keep them terse, keyword-heavy like how real users would search. i.e.:
    - "k8s pod memory limits configuration"
    - "python async await best practices"
    - "quarterly revenue breakdown Q3"
- Queries / answers does not need to encompass the whole chunk. Query just need to target specific piece in the chunk that a user would likely want to know
- Query does not have to completely target all keywords in the chunk since users often only have partial recollection of the information, which is why they are searching
- The query should be answerable from the provided chunk alone
- Paraphrase the keywords / use synonyms in the query to make it natural and varied
- Rank and place your best question first if multiple q&a pairs are generated

Return JSON with:
- keywords: Relevant keywords that a user might search for in the chunk
- confidence: "low" | "mid" | "high";  use "low" if chunk has no meaningful information (too generic, boilerplate, or navigation-only)
- qa_pairs: List of query and answer pairs in the form of [{{"query": "q1", "answer": "a1"}}, ...]
"""

single_hop_system_prompt = render_template(single_hop_system_template, corpus_context)

single_hop_user_template = """Generate a single-hop search q&a pairs based on the following chunk:

<context_before>
{prev_chunk_preview}
</context_before>

<main_chunk>
{chunk_content}
</main_chunk>

<context_after>
{next_chunk_preview}
</context_after>

Context before and after are only provided as additional context. Q&A should only target main chunk content.

Return JSON with keys: keywords, confidence, qa_pairs
"""


def generate_single_hop_qa(
    chunk: Chunk,
    collection: ChunkCollection,
    client: OpenAI,
    model: str,
    context_preview_chars: int,
) -> tuple[str, str, str]:
    # Build the user prompt with chunk and context
    ctx = collection.get_chunk_with_context(chunk, context_max_chars=context_preview_chars)
    single_hop_user_prompt = render_template(single_hop_user_template, ctx)

    # Call the LLM
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": single_hop_system_prompt},
            {"role": "user", "content": single_hop_user_prompt},
        ],
    )

    response = completion.choices[0].message.content or ""

    return single_hop_system_prompt, single_hop_user_prompt, response


# Sample 2 random chunks to inspect synthetic q&a quality
sample_chunks = random.sample(filter_chunks_by_length(collection.chunks, min_chars=400), 2)

for i, chunk_a in enumerate(sample_chunks, 1):
    print(f"{'=' * 120}\nGenerating single-hop Q&A response should take ~10+ seconds...\n")

    system_prompt, user_prompt, response = generate_single_hop_qa(
        chunk_a, collection, client, MODEL, CONTEXT_PREVIEW_CHARS
    )

    confidence, qa_pairs = parse_single_hop_response(response)

    print_sections(
        # (f"SAMPLE {i} of {len(sample_chunks)} - System Prompt", system_prompt),
        (f"SAMPLE {i} of {len(sample_chunks)} - User Prompt", user_prompt),
        (f"SAMPLE {i} of {len(sample_chunks)} - Response", response),
        # (
        #     f"SAMPLE {i} of {len(sample_chunks)} - Parsed Results",
        #     f"Confidence: {confidence}\nQ&A Pairs: {qa_pairs}",
        # ),
    )

### 3.4 Multi-Hop Q&A Generation

Multi-hop questions require information from multiple chunks that share common topics/entities. Here is the process:

1. **Generate search queries** - For a given chunk, generate queries to find related chunks
2. **Search corpus** - Use BM25 to find candidate related chunks
3. **Filter and rank** - Remove adjacent chunks, rank by query overlap and score
4. **Validate connections** - Check if chunks have meaningful dependencies and generate Q&A pairs


In [ ]:
from synthetic_data_prep.qa_generation.helpers import search_related_chunks
from synthetic_data_prep.qa_generation.response_parsers import (
    parse_multi_hop_validation_response,
    parse_related_queries_response,
)

# ===== CONFIGURATION =====
CONTEXT_PREVIEW_CHARS_RELATED = 200

TOP_K_BM25_RESULTS = 5  # Number of top chunks to retrieve for related chunk search
TOP_RELATED_CHUNKS = 3  # Number of related chunks to perform multi-hop qa generation

# =========================


# Prompt for finding related chunks
related_chunk_system_template = """You are generating BM25 search queries to find chunks that have meaningful relationships with the given chunk.

Corpus summary:
{corpus_summary}

## BM25 Behavior
BM25 matches exact keywords, weighted by rarity. This means:
- Specific/rare terms (product names, technical terms, unique phrases) are powerful
- Common corpus terms (e.g., "API", "data", "system") barely help
- BM25 won't match synonyms: "k8s" won't find "Kubernetes"
- Shorter, focused queries often outperform long ones

## Query Strategies

**1. Entity-focused**: Target specific named things that might appear elsewhere
  - Product/tool names: "Redis", "Workday", "Stripe"
  - Internal terms: "Project Atlas", "Q3 planning", "customer churn analysis"
  - Document names: "migration guide", "onboarding checklist"

**2. Reference-chasing**: If this chunk mentions other docs/sections, query for them
  - "see the deployment guide" → query: "deployment guide"
  - "as discussed in Q2 review" → query: "Q2 review"

**3. Inverse references**: Query for terms that other chunks might use to reference this one
  - If this is the Redis setup guide → query: "Redis setup", "Redis configuration"
  - If this covers auth flow → query: "authentication", "OAuth implementation"

**5. Synonym/variant expansion**: Generate alternate phrasings for key concepts
  - "Kubernetes" + "k8s"
  - "authentication" + "auth" + "login"
  - "configuration" + "config" + "setup"

## Query Format
- Prefer specific terms over generic ones
- Include both the canonical term and common variants
- Each query should target a *different* potential related chunk
- If the chunk is boilerplate (e.g., empty template, generic footer), set confidence to "low" and generate few/no queries

Return JSON with:
- keywords: Distinctive terms from this chunk likely to appear in related chunks
- confidence: "low" | "mid" | "high" - based on how much unique, linkable content exists
- queries: ["q1", "q2", ...] - diverse queries targeting different potential relationships
"""

related_chunk_system_prompt = render_template(related_chunk_system_template, corpus_context)

related_chunk_user_template = """Generate search queries based on this chunk to find other relevant chunks:

<context_before>
{prev_chunk_preview}
</context_before>

<main_chunk>
{chunk_content}
</main_chunk>

<context_after>
{next_chunk_preview}
</context_after>

The before/after context is provided only as additional context. Queries should target content from the main chunk only.

Return JSON with keys: keywords, confidence, queries
"""


# Prompt for validating chunk connections and generating QA
multi_hop_system_template = """You are analyzing whether two chunks have a meaningful dependency.

Corpus summary:
{corpus_summary}

Example queries from corpus:
{corpus_queries}

## Task

Analyze two chunks to determine if a meaningful relationship exists, then generate multi-hop questions that exploit that relationship.

**Terminology:**
- **Source chunk**: Contains the reference, pointer, or entry point
- **Target chunk**: Contains the referenced content, details, or destination

## Overall Notes:
- Queries should be terse, keyword-heavy like real user searches: "k8s pod memory limits configuration", "quarterly revenue breakdown Q3"
- Connecting queries come from BM25 (keyword matching)—shared terms don't guarantee meaningful relationships. Scrutinize whether matched terms have the same meaning in both chunks.
- Same high-level entity ≠ valid connection. Two chunks mentioning "Q3 revenue" or "the API" need actual dependency, not just shared subject matter.
- **When in doubt, choose "No Valid Relationship."** Weak relationships produce bad questions.
- **Hard requirements for every question:**
  1. Question vocabulary matches source chunk better than target
  2. Source chunk contains explicit or inferrable path to target
  3. Answer cannot be complete without target chunk's information
  4. Target chunk's distinctive terms must be paraphrased (use hypernyms, describe function/outcome, genericize proper nouns)
- Rank and place your best question first if multiple q&a pairs are generated

## Step 1: Identify relationship type

Classify into exactly one category:

### Explicit Reference
One chunk mentions, names, or links to content that the other chunk *is*.

Signals: Direct references ("see X", "refer to X"), matching document/section names, links, citations, phrases like "as explained in..."

Example: Source says "Q3 planning doc references the customer research findings" → Target is the customer research report

If found: Note direction (A→B means A is source, B is target)

### Abstraction Levels
Same core information at different granularity. One summarizes/claims, the other details/proves.

Signals: Claim + evidence, code + rationale, concept + procedure, summary + full content

Example: Source says "Mediterranean diet has strong evidence for heart health" → Target has study details with participant data and outcomes

Ensure both chunks refer to the same topic/sub-topic, not something tangentially related. **Bidirectional questions possible**—either chunk can serve as source.

### No Valid Relationship

Choose this if:
- Chunks are unrelated or connection is superficial
- Connection requires excessive inference (more than one logical step / only tangentially related)
- Near-duplicates (multi-hop adds no value)
- Content is independently complete on similar subjects

## Step 2: Generate Questions

Skip if relationship type is "No Valid Relationship."

### Question Strategies

**Explicit Reference:** Frame question around the *context* in source where reference appears. Ask for what target provides using paraphrased terms.

Source: "Trip itinerary mentions the restaurant recommendations doc"
Target: [Restaurant list: Tsuta for ramen, Sukiyabashi for sushi...]
✓ "good food japan trip plan" — matches source context, needs target for specifics, no target keywords
✗ "Tsuta ramen Sukiyabashi sushi" — retrieves target directly, bypasses source
✗ "japan trip plan" — matches source, but has no relevance to the target

Source: "Customer onboarding improvements are detailed in the Q3 ops review"
Target: [Ops review data: automated welcome emails, support ticket reduction 60%, average onboarding time 3 days → 4 hours...]
✓ "what changed in customer onboarding process" — matches source context, needs target for specifics, no target keywords
✗ "welcome email automation support ticket reduction" — retrieves target directly, bypasses source
✗ "when was Q3 ops review published" — retrieves source, but asks about metadata, not onboarding details

**Abstraction Levels:** Use vocabulary from source chunk, require precision only target provides. Generate questions in both directions.

General: "The org restructure significantly improved cross-team collaboration"
Specific: [Survey data: cross-team project completion up 40%, meeting conflicts down 25%...]
✓ General→Specific: "measurable impact of reorg on team collaboration"
✓ Specific→General: "leadership claims about collaboration survey results"
✗ "cross-team project completion meeting conflicts" — retrieves specific directly, bypasses general
✗ "org restructure announcement" — retrieves general, but no indication that survey data is needed

General: "Our new caching layer reduced API latency dramatically"
Specific: [Redis implementation: cache hit rates 94%, p99 latency down from 800ms to 120ms, eviction policies...]
✓ General→Specific: "performance metrics for the API speed improvements"
✓ Specific→General: "engineering summary of Redis cache results"
✗ "Redis cache hit rate p99 latency" — retrieves specific directly, bypasses general
✗ "new caching layer" — retrieves general, but no indication that implementation details are needed

Return JSON with:
- thoughts: Analysis of the relationship. State the relationship type found (or why none exists). If relationship exists, identify source vs target and the linking mechanism.
- relationship_type: "explicit_reference" | "abstraction_levels" | "none"
- direction: "A_to_B" | "B_to_A" | "bidirectional" | null
- linking_info: Object describing the connection, or null if none. Structure depends on relationship_type:
    - For explicit_reference: {{ reference_text, source ("A"|"B"), target ("A"|"B") }}
    - For abstraction_levels: {{ general_chunk ("A"|"B"), specific_chunk ("A"|"B"), abstraction_link }}
- qa_pairs: List of QA pairs, or null if no valid multi-hop questions can be formed. Each pair contains:
    - question: Terse, keyword-focused multi-hop question
    - answer: Answer requiring synthesis of both chunks
"""

multi_hop_system_prompt = render_template(multi_hop_system_template, corpus_context)

multi_hop_user_template = """Analyze the connection between these chunks.

Connecting Queries: {connecting_queries}

<chunk_a_context_before>
{chunk_a_context_before}
</chunk_a_context_before>

<chunk_a>
{chunk_a}
</chunk_a>

<chunk_a_context_after>
{chunk_a_context_after}
</chunk_a_context_after>

---

<chunk_b_context_before>
{chunk_b_context_before}
</chunk_b_context_before>

<chunk_b>
{chunk_b}
</chunk_b>

<chunk_b_context_after>
{chunk_b_context_after}
</chunk_b_context_after>

Analyze whether there is a meaningful relationship between the chunks and whether multi-hop questions can be formed.

Return JSON with keys: thoughts, relationship_type, direction, linking_info, qa_pairs
"""


def generate_related_queries(
    chunk: Chunk,
    collection: ChunkCollection,
    client: OpenAI,
    model: str,
    context_preview_chars: int = 200,
) -> tuple[str, str, str]:
    """Generate queries to find related chunks. Returns (queries, parsed_result)."""
    ctx = collection.get_chunk_with_context(chunk, context_max_chars=context_preview_chars)
    user_prompt = render_template(related_chunk_user_template, ctx)

    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": related_chunk_system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    response_text = completion.choices[0].message.content or ""
    return related_chunk_system_prompt, user_prompt, response_text


def generate_multi_hop_qa(
    chunk_a: Chunk,
    chunk_b: Chunk,
    collection: ChunkCollection,
    connecting_queries: list[str],
    client: OpenAI,
    model: str,
    context_preview_chars: int = 200,
) -> tuple[str, str, str]:
    """Validate connection and generate QA pairs. Returns (system_prompt, user_prompt, response_text)."""
    # Get context for both chunks
    ctx_a = collection.get_chunk_with_context(chunk_a, context_max_chars=context_preview_chars)
    ctx_b = collection.get_chunk_with_context(chunk_b, context_max_chars=context_preview_chars)

    user_prompt = render_template(
        multi_hop_user_template,
        {
            "connecting_queries": connecting_queries,
            "chunk_a": ctx_a["chunk_content"],
            "chunk_a_context_before": ctx_a["prev_chunk_preview"],
            "chunk_a_context_after": ctx_a["next_chunk_preview"],
            "chunk_b": ctx_b["chunk_content"],
            "chunk_b_context_before": ctx_b["prev_chunk_preview"],
            "chunk_b_context_after": ctx_b["next_chunk_preview"],
        },
    )

    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": multi_hop_system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    response_text = completion.choices[0].message.content or ""
    return multi_hop_system_prompt, user_prompt, response_text


# Sample 2 chunks to test related chunk finding and multi-hop QA generation
sample_chunks = random.sample(filter_chunks_by_length(collection.chunks, min_chars=400), 2)

for i, chunk_a in enumerate(sample_chunks, 1):
    print_sections((f"Chunk {i}, {len(chunk_a)} chars", chunk_a.chunk_str()))
    print(
        f"{'=' * 120}\nFinding related queries for SAMPLE {i} of {len(sample_chunks)}. This should take ~10s+..."
    )

    system_prompt_step_1, user_prompt_step_1, response = generate_related_queries(
        chunk_a, collection, client, MODEL, CONTEXT_PREVIEW_CHARS_RELATED
    )

    confidence, queries = parse_related_queries_response(response)

    # # UNCOMMENT to print step 1 details
    # print_sections(
    #     # (f"SAMPLE {i} of {len(sample_chunks)} - Step 1 System Prompt", system_prompt_step_1),
    #     (f"SAMPLE {i} of {len(sample_chunks)} - Step 1 User Prompt", user_prompt_step_1),
    #     (f"SAMPLE {i} of {len(sample_chunks)} - Step 1 Response", response),
    #     (f"SAMPLE {i} of {len(sample_chunks)} - Parsed Related Queries",
    #         f"Confidence: {confidence}\nQueries: {queries}",
    #     ),
    # )

    if confidence.lower() != "high":
        print("Skipping multi-hop generation due to low confidence in related queries.\n")
        continue

    # Search for related chunks
    search_results = search_related_chunks(
        chunk_a, queries, corpus_client, corpus, collection, TOP_K_BM25_RESULTS
    )

    # Take the top related chunk
    for j, chunk_info in enumerate(search_results[:TOP_RELATED_CHUNKS], 1):
        chunk_b = chunk_info["chunk"]
        connecting_queries = chunk_info["queries"]

        print(
            f"{'-' * 120}\nGenerating multi-hop Q&A pair for SAMPLE {i}.{j} of {len(sample_chunks)}. This might take ~20s+..."
        )

        (
            system_prompt_step_2,
            user_prompt_step_2,
            response,
        ) = generate_multi_hop_qa(chunk_a, chunk_b, collection, connecting_queries, client, MODEL)

        qa_pairs = parse_multi_hop_validation_response(response)

        if not qa_pairs:
            print("No QA pairs generated.\n")
            continue

        print_sections(
            # (f"SAMPLE {i}.{j} of {len(sample_chunks)} - Step 2 System Prompt", system_prompt_step_2),
            (f"SAMPLE {i}.{j} of {len(sample_chunks)} - Step 2 User Prompt", user_prompt_step_2),
            (f"SAMPLE {i}.{j} of {len(sample_chunks)} - Step 2 Response", response),
            # (
            #     f"SAMPLE {i}.{j} of {len(sample_chunks)} - Parsed Multi-hop QA",
            #     f"Generated {len(qa_pairs)} QA pairs: {qa_pairs}",
            # ),
        )

## 4. Batch Run

This section will process all chunks in batch to generate the full dataset.

**Configuration:**

- Number of single-hop request -> each request might generate 0-2 questions.
- Number of multi-hop request -> each request might generate 0-10 questions.


In [ ]:
from synthetic_data_prep.qa_generation.helpers import (
    generate_multi_hop_batch,
    generate_single_hop_batch,
)
from synthetic_data_prep.qa_generation.storage import (
    qa_dataset_to_jsonl_bytes,
    save_qa_dataset,
    save_qa_dataset_jsonl,
)

# ===== CONFIGURATION =====
NUM_SINGLE_HOP_SAMPLES = 40
NUM_MULTI_HOP_SAMPLES = 5

# Maximum number of questions to use per chunk
# Set to None for no limit (use all generated questions)
MAX_QUESTIONS_PER_CHUNK = 2  # Set to 2 to get more chunk diversity

# Batch processing configuration
MAX_CONCURRENT = 10  # Number of concurrent LLM calls
MAX_TOKENS = 1000  # Max tokens per response
TIMEOUT = 60.0  # Request timeout in seconds

# Output paths
OUTPUT_PATH_YAML = "outputs/qa_dataset.yaml"  # Human-readable format
OUTPUT_PATH_JSONL = "outputs/qa_dataset.jsonl"  # HuggingFace-compatible format
# =========================


# Generate single-hop QA pairs in batch
print(f"\n{'=' * 120}\nGenerating single-hop QA pairs...\n{'=' * 120}")

single_hop_dataset = generate_single_hop_batch(
    collection=collection,
    client=client,
    model=MODEL,
    system_prompt=single_hop_system_prompt,
    user_template=single_hop_user_template,
    num_samples=NUM_SINGLE_HOP_SAMPLES,
    response_parser=parse_single_hop_response,
    context_preview_chars=CONTEXT_PREVIEW_CHARS,
    max_concurrent=MAX_CONCURRENT,
    max_tokens=MAX_TOKENS,
    timeout=TIMEOUT,
    max_questions=MAX_QUESTIONS_PER_CHUNK,
)

print(f"\n{single_hop_dataset.summary()}")

# Generate multi-hop QA pairs in batch
print(f"\n{'=' * 120}\nGenerating multi-hop QA pairs...\n{'=' * 120}")

multi_hop_dataset = generate_multi_hop_batch(
    collection=collection,
    client=client,
    model=MODEL,
    related_query_system_prompt=related_chunk_system_prompt,
    related_query_user_template=related_chunk_user_template,
    multi_hop_system_prompt=multi_hop_system_prompt,
    multi_hop_user_template=multi_hop_user_template,
    corpus_client=corpus_client,
    corpus=corpus,
    num_samples=NUM_MULTI_HOP_SAMPLES,
    related_query_parser=parse_related_queries_response,
    multi_hop_parser=parse_multi_hop_validation_response,
    context_preview_chars=CONTEXT_PREVIEW_CHARS_RELATED,
    top_k_bm25=TOP_K_BM25_RESULTS,
    top_related_chunks=TOP_RELATED_CHUNKS,
    max_concurrent=MAX_CONCURRENT,
    max_tokens=MAX_TOKENS,
    timeout=TIMEOUT,
    max_questions=MAX_QUESTIONS_PER_CHUNK,
)

print(f"\n{multi_hop_dataset.summary()}")

# Merge datasets and save
print(f"\n{'=' * 120}\nMerging and saving... {'=' * 120}")

combined_dataset = single_hop_dataset.merge(multi_hop_dataset)
print(f"\n{combined_dataset.summary()}")

# Save in both formats
save_qa_dataset(combined_dataset, OUTPUT_PATH_YAML)
print(f"\nSaved YAML (human-readable) to {OUTPUT_PATH_YAML}")

save_qa_dataset_jsonl(combined_dataset, OUTPUT_PATH_JSONL)
print(f"Saved JSONL (HuggingFace-compatible) to {OUTPUT_PATH_JSONL}")

# # UNCOMMENT the code below to load the saved dataset
# loaded_dataset = load_qa_dataset(OUTPUT_PATH_YAML)
# print(f"\nLoaded dataset: {loaded_dataset.summary()}")

## 5. Upload Q&A Dataset

Upload the generated Q&A pairs to storage for model training and evaluation.

The dataset is uploaded in JSONL format (HuggingFace-compatible) to the trainer mounted storage.


In [ ]:
from synthetic_data_prep.trainer import StorageClient

# ===== MODIFY HERE: Configure storage settings =====
# Note: You can use a different base URL if storage endpoint differs
STORAGE_BASE_URL = CORPUS_BASE_URL
# ===== END OF MODIFY SECTION =====

storage_client = StorageClient(api_key=API_KEY, base_url=STORAGE_BASE_URL)

# Upload the generated dataset as JSONL
print(f"Uploading {len(combined_dataset)} QA pairs for corpus {corpus.id}...")

dataset_path = f"datasets/search/{corpus.id}/qa-dataset.jsonl"

try:
    result = storage_client.upload_file(
        path=dataset_path,
        content=qa_dataset_to_jsonl_bytes(combined_dataset),
        mime_type="application/jsonl",
    )
    print("\nUpload successful!")
    print(f"  Blob path: {result['blobPath']}")
except Exception as e:
    print(f"\nUpload failed: {e}")

# # UNCOMMENT to upload from a saved file instead
# try:
#     result = storage_client.upload_local_file(
#         path=dataset_path,
#         file_path=OUTPUT_PATH_JSONL,
#     )
#     print("\nUpload successful!")
#     print(f"  Blob path: {result['blobPath']}")
# except Exception as e:
#     print(f"\nUpload failed: {e}")

## 5. RL Environment

This is the search environment that the trained LLM will be interacting with via tool calls. In an environment, we define the system prompt, tools, and reward functions. The reward function is executed at the end of the rollout (when the model reports the answer) and the score will be used to optimize the model

For the search environment, our tool is performing BM25 search on the corpus. (TODO: Need to implement llm judge rubric + reference chunk overlap calc) Our reward function ...


In [ ]:
from collections.abc import Callable
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any

import aiohttp
from benchmax.envs.base_env import BaseEnv
from benchmax.envs.types import StandardizedExample, ToolDefinition

SYSTEM_PROMPT = """Please use the search tool provided to find relevant information from the corpus.
Formulate effective search queries to retrieve the most relevant chunks.
You can filter by metadata or filename to narrow your search.
Write your complete answer on the final line only as a concise entity, within the xml tags <answer></answer>.\n
"""


def percent_of_text_a_in_text_b(text_a, text_b):
    if not text_a:
        return 0.0

    matcher = SequenceMatcher(None, text_a, text_b)
    matched_chars = sum(size for _, _, size in matcher.get_matching_blocks())
    return matched_chars / len(text_a)


async def chunk_overlap_reward_function(completion: str, ground_truth: str, **kwargs: Any) -> float:
    """
    Reward function that computes the percentage of overlapping text between
    the completion and the ground truth.

    Args:
        completion: The model's generated text
        ground_truth: The reference text to compare against
        **kwargs: Additional arguments (not used here)
    Returns:
        float: A score between 0.0 and 1.0 representing the overlap percentage.
    """
    reference_chunks = kwargs.get("reference_chunks", [])
    reference_string = " ".join(
        [
            chunk.get("content", "") if isinstance(chunk, dict) else str(chunk)
            for chunk in reference_chunks
        ]
    )
    completion_str = completion if isinstance(completion, str) else ""
    if isinstance(completion, list):
        completion_str = " ".join(
            [
                c.get("content", "")
                for c in completion
                if isinstance(c, dict) and c.get("role", "") != "assistant"
            ]
        )
        for msg in completion:
            if not isinstance(msg, dict):
                continue
            if msg.get("role", "") != "assistant":
                continue
            msg_content = msg.get("content", "")
            if msg_content.count("<tool_call>") >= 4:
                return 0.0

    if reference_string:
        overlap_score = percent_of_text_a_in_text_b(reference_string, completion_str)
        if overlap_score >= 0.25:
            return overlap_score
    return 0.0


class SearchEnv(BaseEnv):
    """Search environment with BM25 corpus search tool."""

    system_prompt: str = SYSTEM_PROMPT

    def __init__(
        self,
        api_key: str,
        corpus_id: str,
        base_url: str,
        dataset_path: str,
        **kwargs,
    ):
        """
        Initialize the search environment.
        """
        self._api_key = api_key
        self._corpus_id = corpus_id
        self._base_url = base_url.rstrip("/")
        self._dataset_path = dataset_path

        search_tool_definition = ToolDefinition(
            name="search_corpus",
            description="Search the corpus using BM25 with optional metadata and filename filtering.",
            input_schema={
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query string.",
                    },
                    "metadata": {
                        "type": "object",
                        "description": "Optional metadata filters (e.g., {'ticker': 'DDOG', 'year': 2024}).",
                    },
                    "filename": {
                        "type": "string",
                        "description": "Optional filename filter. Simple string for substring match (e.g., 'config') or regex pattern (e.g., '.*\\.json$').",
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Max number of results to return (default 10).",
                    },
                },
                "required": ["query"],
            },
        )

        self._tools: dict[str, tuple[ToolDefinition, Callable]] = {
            search_tool_definition.name: (search_tool_definition, self._search_corpus_tool)
        }

    async def _search_corpus_tool(
        self,
        query: str,
        metadata: dict[str, Any] | None = None,
        filename: str | None = None,
        limit: int = 10,
        **kwargs,
    ) -> str:
        """
        Search the corpus using BM25.

        Args:
            query: Search query string
            metadata: Optional metadata filters
            filename: Optional filename filter (substring or regex)
            limit: Maximum number of results

        Returns:
            Formatted search results or error message
        """
        if not query:
            return "Error: Missing required parameter: 'query'"

        # Build request body
        request_body = {"query": query, "limit": limit}
        if metadata:
            request_body["metadata"] = metadata
        if filename:
            request_body["filename"] = filename

        # Build URL
        url = f"{self._base_url}/api/corpora/{self._corpus_id}/search"
        headers = {
            "Authorization": f"Bearer {self._api_key}",
            "Content-Type": "application/json",
        }

        try:
            async with aiohttp.ClientSession() as session:
                async with session.post(
                    url,
                    json=request_body,
                    headers=headers,
                    timeout=aiohttp.ClientTimeout(total=10.0),
                ) as resp:
                    if resp.status != 200:
                        error_text = await resp.text()
                        return f"Error: API request failed with status {resp.status}: {error_text}"

                    data = await resp.json()

            results = data.get("results", [])
            total = data.get("total", 0)

            if not results:
                return "No results found."

            # Format results
            lines = []
            for i, item in enumerate(results, start=1):
                filename_val = item.get("filename", "—")
                score = item.get("score")
                score_str = f"(score: {score:.2f})" if score is not None else "(filtered)"
                content = item.get("content", "")
                metadata_val = item.get("metadata", {})

                lines.append(f"{i}. {filename_val} {score_str}")
                lines.append(f"   Content: {content}")
                if metadata_val:
                    lines.append(f"   Metadata: {metadata_val}")

            lines.append(f"\nTotal: {total} results")
            return "\n".join(lines)

        except aiohttp.ClientError as e:
            return f"Error: Network error: {str(e)}"
        except Exception as e:
            return f"Error: {str(e)}"

    def get_train_val_split(self, train_ratio: float = 0.7, seed: int = 42, **kwargs):
        """Load the dataset and return stratified train/validation splits by qa_type.

        Args:
            train_ratio: Fraction of data to use for training (default 0.7 for 70-30 split)
            seed: Random seed for reproducibility
            **kwargs: Additional arguments passed to load_dataset

        Returns:
            Tuple of (train_dataset, val_dataset) with stratified splits by qa_type
        """
        from datasets import concatenate_datasets
        from datasets import load_dataset as hf_load_dataset
        # These deps are already included when installing benchmax, so we didn't have to add it to the pip deps

        ds = hf_load_dataset(
            "json", data_files=str(Path(self._dataset_path).expanduser().absolute())
        )
        dataset = ds["train"]

        # Get unique qa_types
        qa_types = set(dataset["qa_type"])

        train_splits = []
        val_splits = []

        for qa_type in qa_types:
            # Filter dataset by qa_type
            type_indices = [i for i, t in enumerate(dataset["qa_type"]) if t == qa_type]
            type_subset = dataset.select(type_indices)

            # Shuffle and split
            type_subset = type_subset.shuffle(seed=seed)
            split_idx = int(len(type_subset) * train_ratio)

            # Handle edge case where split would be empty
            if split_idx == 0 and len(type_subset) > 0:
                split_idx = 1
            elif split_idx == len(type_subset) and len(type_subset) > 1:
                split_idx = len(type_subset) - 1

            train_splits.append(type_subset.select(range(split_idx)))
            val_splits.append(type_subset.select(range(split_idx, len(type_subset))))

        # Concatenate all splits
        train_dataset = concatenate_datasets(train_splits)
        val_dataset = concatenate_datasets(val_splits)

        # Shuffle the final datasets to mix qa_types
        train_dataset = train_dataset.shuffle(seed=seed)
        val_dataset = val_dataset.shuffle(seed=seed)

        return train_dataset, val_dataset

    @classmethod
    def dataset_preprocess(cls, example: Any, **kwargs) -> StandardizedExample:
        return StandardizedExample(
            prompt=example.get("question", ""),
            ground_truth=example.get("answer", None),
            init_rollout_args={},
        )

    async def list_tools(self) -> list[ToolDefinition]:
        """List available tools."""
        return [self._tools[k][0] for k in sorted(self._tools)]

    async def run_tool(self, rollout_id: str, tool_name: str, **tool_args) -> Any:
        """
        Execute a tool.

        Args:
            rollout_id: Identifier for current rollout (unused for stateless env)
            tool_name: Name of the tool (e.g., "search_corpus")
            **tool_args: Arguments for the tool function

        Returns:
            Tool execution result or error message
        """
        _, tool_function = self._tools[tool_name]
        return await tool_function(**tool_args)

    reward_funcs = [chunk_overlap_reward_function]

    async def compute_reward(
        self, rollout_id: str, completion: str, ground_truth: Any, **kwargs: Any
    ) -> dict[str, float]:
        """Compute rewards using the chunk overlap reward function."""
        return {
            "chunk_overlap_reward_function": await chunk_overlap_reward_function(
                completion, ground_truth, **kwargs
            )
        }

    async def init_rollout(self, rollout_id: str, **rollout_args) -> None:
        """Initialize rollout (no-op for stateless environment)."""
        pass

    async def release_rollout(self, rollout_id: str) -> None:
        """Release rollout (no-op for stateless environment)."""
        pass

    async def copy_to_workspace(
        self, rollout_id: str, src_path: Path, dst_filename: str | None = None
    ) -> None:
        """Not needed for this environment."""
        pass

    async def copy_content_to_workspace(
        self, rollout_id: str, src_content: str | bytes, dst_filename: str
    ) -> None:
        """Not needed for this environment."""
        pass

    async def copy_from_workspace(self, rollout_id: str, src_filename: str, dst_path: Path) -> None:
        """Not needed for this environment."""
        pass

    async def shutdown(self):
        # no cleanup required
        pass

Now that we have defined our search environment, let's bundle it and upload to the trainer storage.


In [ ]:
import hashlib
import json

from benchmax.bundle.bundler import bundle_env
from benchmax.bundle.validator import validate_payload

constructor_args = {
    "api_key": API_KEY,
    "corpus_id": corpus.id,
    "base_url": "https://app.cgft.io",
    "dataset_path": f"~/user-data/{dataset_path}",
    # include ~/user-data/ prefix to index since that is where your storage would be mounted
}

payload = bundle_env(
    SearchEnv,
    pip_dependencies=["aiohttp"],  # Dependencies required for search functionality
    local_modules=[],  # No local modules to package
    extra_metadata={"description": "Search env with local helpers"},
)

print(f"Payload size: {len(payload.to_bytes()) / 1024:.2f} KB")
print(f"Python version: {payload.python_version}")
print(f"Dependencies: {payload.pip_dependencies}")


# TODO: Make this run the full eval loop to test the behavior e2e
print(
    "\nRunning validation in fresh environment to ensure all dependencies are correctly declared."
    "\nThis will take ~1 min..."
)
warnings = validate_payload(
    payload,
    constructor_args=constructor_args,
)
if warnings:
    print(f"Warnings: {warnings}")
else:
    print("Isolated validation passed!")

# Upload env
payload_bytes = payload.to_bytes()

# Calculate hash from content
content_hash = hashlib.sha256(payload_bytes).hexdigest()[:8]

# Upload to storage with hash in filename
env_path = f"envs/search/{content_hash}/search-env-cls.bmxp"
env_result = storage_client.upload_file(
    path=env_path,
    content=payload_bytes,
    mime_type="application/octet-stream",
)

env_args_path = f"envs/search/{content_hash}/search-env-kwargs.json"
env_args_bytes = json.dumps(constructor_args, indent=2).encode("utf-8")
env_args_result = storage_client.upload_file(
    path=env_args_path,
    content=env_args_bytes,
    mime_type="application/json",
)

print(f"Env uploaded successfully to {env_result['blobPath']}")
print(f"Env args uploaded successfully to {env_args_result['blobPath']}")


## 6. Start Training Job

Launch a training job using the uploaded dataset.


In [ ]:
from synthetic_data_prep.trainer import TrainerClient

# ===== MODIFY HERE: Configure trainer settings =====
TRAINER_API_KEY = API_KEY
TRAINER_BASE_URL = CORPUS_BASE_URL
# ===== END OF MODIFY SECTION =====

trainer_client = TrainerClient(api_key=TRAINER_API_KEY, base_url=TRAINER_BASE_URL)

# Launch training job with the uploaded dataset
try:
    job = trainer_client.launch_job(
        job_type="search",
        args={"dataset": result["blobPath"]},
    )
    print("Training job launched!")
    print(f"  Job response: {job}")
except Exception as e:
    print(f"\nFailed to launch training job: {e}")
